# 4. Testing the trained model on new ISL video

```text
New ISL Video -> Extract Frames -> MediaPipe -> Landmarks
              -> Resample (64 frames) -> Normalize (shoulder-relative)
              -> Trained LSTM -> Predicted ISL Sign -> English Word
```

**Important:** this model recognizes ONE isolated sign at a time. It does
not understand grammar or continuous signing.

## 0. Setup (Colab or local)

In [ ]:
import os

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/ISL_Sign2Sub"
    ON_COLAB = True
except ImportError:
    PROJECT_DIR = os.path.abspath("..")
    ON_COLAB = False

MODEL_PATH = os.path.join(PROJECT_DIR, "model", "sign_model.pth")
CLASS_NAMES_PATH = os.path.join(PROJECT_DIR, "model", "class_names.json")

print("Running on Colab:", ON_COLAB)
if not os.path.exists(MODEL_PATH):
    print("No trained model found yet. Run 3_train.ipynb first.")
else:
    print("Found trained model at", MODEL_PATH)

## 1. Install MediaPipe and download the detector models

In [ ]:
!pip install -q mediapipe
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task

## 2. Load everything: MediaPipe, preprocessing functions, and the trained model

These functions must exactly match the ones used in `2_landmarks.ipynb` and
`3_train.ipynb` (feature extraction, resampling, normalization), or the
model receives input it was never trained on.

In [ ]:
import json, cv2, numpy as np, torch, torch.nn as nn
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BaseOptions = mp.tasks.BaseOptions
RunMode = mp.tasks.vision.RunningMode
hand_landmarker = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=RunMode.IMAGE, num_hands=2))
pose_landmarker = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="pose_landmarker.task"),
    running_mode=RunMode.IMAGE))

def frame_to_mp_image(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

def extract_frame_features(frame_bgr):
    mp_image = frame_to_mp_image(frame_bgr)
    hand_result = hand_landmarker.detect(mp_image)
    pose_result = pose_landmarker.detect(mp_image)
    left, right = np.zeros(63, np.float32), np.zeros(63, np.float32)
    pose = np.zeros(99, np.float32)
    for lms, hd in zip(hand_result.hand_landmarks, hand_result.handedness):
        coords = np.array([[l.x, l.y, l.z] for l in lms], np.float32).flatten()
        if hd[0].category_name == "Left": left = coords
        else: right = coords
    if pose_result.pose_landmarks:
        pose = np.array([[l.x, l.y, l.z] for l in pose_result.pose_landmarks[0]], np.float32).flatten()
    return np.concatenate([left, right, pose])

def read_frames(path):
    cap = cv2.VideoCapture(path); out = []
    while True:
        ok, fr = cap.read()
        if not ok: break
        out.append(fr)
    cap.release()
    return out

def resample_sequence(seq, target_len):
    idx = np.linspace(0, len(seq) - 1, target_len).round().astype(int)
    return seq[idx]

def normalize_sequences(X):
    N, T, _ = X.shape
    pose = X[:, :, 126:].reshape(N, T, 33, 3)
    ls, rs = pose[:, :, 11, :], pose[:, :, 12, :]
    center = (ls + rs) / 2
    scale = np.linalg.norm((ls - rs)[:, :, :2], axis=2, keepdims=True)
    scale = np.where(scale < 1e-3, 1.0, scale)
    out = X.copy()
    for start in (0, 63):
        hand = X[:, :, start:start + 63].reshape(N, T, 21, 3)
        missing = np.all(hand == 0, axis=(2, 3), keepdims=True)
        hand_n = (hand - center[:, :, None, :]) / scale[:, :, None, :]
        out[:, :, start:start + 63] = np.where(missing, 0.0, hand_n).reshape(N, T, 63)
    out[:, :, 126:] = ((pose - center[:, :, None, :]) / scale[:, :, None, :]).reshape(N, T, 99)
    return out.astype(np.float32)

class SignLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_classes=10):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        return self.fc(h_n[-1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(MODEL_PATH, map_location=device)
class_names = json.load(open(CLASS_NAMES_PATH))
model = SignLSTM(ckpt["input_size"], ckpt["hidden_size"], len(class_names)).to(device)
model.load_state_dict(ckpt["state_dict"]); model.eval()
TARGET_LEN = ckpt["target_len"]
print("Model loaded. Classes:", list(class_names.values()))

## 3. Predict a single video

In [ ]:
def predict_video(video_path):
    frames = read_frames(video_path)
    seq = np.stack([extract_frame_features(f) for f in frames])
    seq = resample_sequence(seq, TARGET_LEN)
    seq = normalize_sequences(seq[None])                 # add batch dimension
    with torch.no_grad():
        probs = torch.softmax(model(torch.tensor(seq).to(device)), dim=1)[0].cpu().numpy()
    top = probs.argsort()[::-1][:3]
    return [(class_names[str(i)], float(probs[i])) for i in top]

test_path = os.path.join(PROJECT_DIR, "data", "videos", "HELLO", "MVI_0029.MOV")
for name, p in predict_video(test_path):
    print(f"{name}: {p:.1%}")

**Note:** a video that was in the training set only proves the pipeline
works end to end — it does not measure real accuracy. For that, use videos
from the held-out test set, or record your own.

## 4. Test on several different signs

Try a few more, swapping the sign folder and filename each time:
```python
test_path = os.path.join(PROJECT_DIR, "data", "videos", "THANK_YOU", "MVI_9948.MOV")
for name, p in predict_video(test_path):
    print(f"{name}: {p:.1%}")
```

## 5. Test on a video of yourself (the real test)

Record yourself doing one of the 8 trained signs, upload it to Colab (the
folder icon in the left sidebar, or via Drive), then:
```python
test_path = "/content/your_video.mp4"
for name, p in predict_video(test_path):
    print(f"{name}: {p:.1%}")
```
Expect lower accuracy than on the dataset's own videos — your camera angle,
lighting, and signing style differ from the dataset's signers.